In [ ]:
# Search pipeline to get workflow performance data for instrumentation tests runs

In [ ]:
# ============================================================
# PASS2-API (No-clone) Pipeline — last 60 days GH Actions performance
# - Reads workflow candidates from CSVs (no repo cloning)
# - Verifies workflows via GitHub API
# - Fetches workflow runs + jobs + steps for last N days
# - Computes simple performance metrics
# ============================================================

import os
import re
import time
import math
import json
import csv
import random
import logging
import datetime as dt
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import requests
import pandas as pd


# -----------------------------
# CONFIG (edit these in notebook)
# -----------------------------
CI_CSV_PATH = r"1_CI_YML_Instru.csv"
GMD_CSV_PATH = r"1_Gradle_GMD_Instru.csv"

LOOKBACK_DAYS = 60
PER_PAGE = 100

# Optional: if you want to load token(s) from a file like tokens.env
# If the file doesn't exist, the script still works if env vars exist.
TOKENS_ENV_PATH = r"tokens.env"

# Output folder
OUTPUT_DIR = r"pass2_api_outputs_60d"

# Optional safety limits (avoid huge downloads)
MAX_RUNS_PER_WORKFLOW = None   # e.g. 500, or None for unlimited
MAX_REPOS = None              # e.g. 200, or None

# Logging
LOG_LEVEL = logging.INFO


# -----------------------------
# Helpers
# -----------------------------
def setup_logger(out_dir: str) -> logging.Logger:
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = Path(out_dir) / f"pass2_api_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    logger = logging.getLogger("pass2_api")
    logger.setLevel(LOG_LEVEL)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    fh = logging.FileHandler(str(log_path), encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.info(f"Logging to {log_path}")
    return logger


def parse_iso_z(s: Optional[str]) -> Optional[dt.datetime]:
    if not s or not isinstance(s, str):
        return None
    # GitHub uses ISO with Z
    try:
        return dt.datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        return None


def to_minutes(seconds: Optional[float]) -> Optional[float]:
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return None
    return seconds / 60.0


def safe_float(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None


def load_env_file(path: str) -> Dict[str, str]:
    """
    Minimal .env loader: KEY=VALUE lines, ignores comments/blank lines.
    Does NOT recurse. Does NOT crash if file missing.
    """
    p = Path(path)
    if not p.exists():
        return {}

    out = {}
    for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k:
            out[k] = v
    return out


def collect_tokens(tokens_env_path: str = TOKENS_ENV_PATH) -> List[str]:
    """
    Accepted sources (first found wins, but we also combine):
      - environment: GITHUB_TOKENS="t1,t2,..." or GITHUB_TOKEN="t1"
      - tokens.env: same keys, or GITHUB_TOKEN_1, GITHUB_TOKEN_2, ...
    """
    env_from_file = load_env_file(tokens_env_path)

    # Merge file env into process env (process env wins if already set)
    for k, v in env_from_file.items():
        os.environ.setdefault(k, v)

    tokens: List[str] = []

    if os.getenv("GITHUB_TOKENS"):
        tokens += [t.strip() for t in os.getenv("GITHUB_TOKENS").split(",") if t.strip()]

    if os.getenv("GITHUB_TOKEN"):
        tokens.append(os.getenv("GITHUB_TOKEN").strip())

    # Also support numbered tokens in tokens.env
    numbered = []
    for k, v in os.environ.items():
        if re.fullmatch(r"GITHUB_TOKEN_\d+", k) and v.strip():
            numbered.append((k, v.strip()))
    for _, v in sorted(numbered, key=lambda kv: kv[0]):
        tokens.append(v)

    # De-dup while preserving order
    seen = set()
    deduped = []
    for t in tokens:
        if t and t not in seen:
            seen.add(t)
            deduped.append(t)

    return deduped


# -----------------------------
# GitHub API client (token rotation + pagination)
# -----------------------------
class GitHubClient:
    def __init__(self, tokens: List[str], logger: logging.Logger):
        if not tokens:
            raise RuntimeError(
                "No GitHub token found.\n"
                "Set environment variable GITHUB_TOKEN (or GITHUB_TOKENS) "
                "or create tokens.env with GITHUB_TOKEN=... (recommended)."
            )
        self.tokens = tokens
        self.idx = 0
        self.logger = logger
        self.session = requests.Session()
        self.base = "https://api.github.com"

    def _headers(self) -> Dict[str, str]:
        token = self.tokens[self.idx]
        return {
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "pass2-api-pipeline",
        }

    def _rotate(self):
        if len(self.tokens) > 1:
            self.idx = (self.idx + 1) % len(self.tokens)

    def request(self, method: str, url: str, params: Optional[Dict[str, Any]] = None, max_retries: int = 6) -> requests.Response:
        backoff = 1.5
        for attempt in range(1, max_retries + 1):
            r = self.session.request(method, url, headers=self._headers(), params=params, timeout=60)

            # Rate limit / secondary rate limit
            if r.status_code == 403:
                msg = ""
                try:
                    msg = r.json().get("message", "")
                except Exception:
                    msg = r.text[:200]

                remaining = r.headers.get("X-RateLimit-Remaining")
                reset = r.headers.get("X-RateLimit-Reset")
                if remaining == "0" or "rate limit" in msg.lower():
                    if len(self.tokens) > 1:
                        self.logger.warning("Rate limit hit; rotating token...")
                        self._rotate()
                        continue

                    # single token: sleep until reset
                    if reset and reset.isdigit():
                        reset_dt = dt.datetime.fromtimestamp(int(reset), tz=dt.timezone.utc)
                        now = dt.datetime.now(tz=dt.timezone.utc)
                        sleep_s = max(5, int((reset_dt - now).total_seconds()) + 3)
                        self.logger.warning(f"Rate limit hit; sleeping {sleep_s}s until reset...")
                        time.sleep(sleep_s)
                        continue

                # Not rate limit; return for caller to handle
                return r

            # Retry transient errors
            if r.status_code in (500, 502, 503, 504):
                sleep_s = min(30, int((backoff ** attempt) + random.random()))
                self.logger.warning(f"GitHub {r.status_code}; retrying in {sleep_s}s (attempt {attempt}/{max_retries})")
                time.sleep(sleep_s)
                continue

            return r

        return r

    def get_json(self, path: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        url = path if path.startswith("http") else f"{self.base}{path}"
        r = self.request("GET", url, params=params)
        if r.status_code >= 400:
            raise RuntimeError(f"GitHub API error {r.status_code} for {url}: {r.text[:500]}")
        return r.json()

    def get_paginated_items(self, path: str, item_key: str, params: Optional[Dict[str, Any]] = None, hard_limit: Optional[int] = None) -> List[Dict[str, Any]]:
        """
        Paginates using Link headers (recommended for GitHub).
        item_key is the JSON key containing list items (e.g. 'workflows', 'workflow_runs', 'jobs').
        """
        url = path if path.startswith("http") else f"{self.base}{path}"
        items: List[Dict[str, Any]] = []
        params = dict(params or {})
        params.setdefault("per_page", PER_PAGE)
        page_url = url

        while True:
            r = self.request("GET", page_url, params=params)
            if r.status_code == 404:
                return items
            if r.status_code >= 400:
                raise RuntimeError(f"GitHub API error {r.status_code} for {page_url}: {r.text[:500]}")

            data = r.json()
            batch = data.get(item_key, [])
            if isinstance(batch, list):
                items.extend(batch)

            if hard_limit is not None and len(items) >= hard_limit:
                return items[:hard_limit]

            # parse Link header
            link = r.headers.get("Link", "")
            next_url = None
            if link:
                parts = [p.strip() for p in link.split(",")]
                for p in parts:
                    if 'rel="next"' in p:
                        m = re.search(r"<([^>]+)>", p)
                        if m:
                            next_url = m.group(1)
                        break

            if not next_url:
                break

            page_url = next_url
            params = None  # next_url already contains query

        return items


# -----------------------------
# Workflow candidate extraction from CSV
# -----------------------------
def workflow_stem_from_candidate_filename(candidate_filename: str) -> str:
    """
    CSV candidate filename looks like:
      <owner>__<repo>__github_actions++main__7.yaml
    We extract 'main' (stem after github_actions++).
    """
    s = str(candidate_filename)
    m = re.search(r"github_actions\+\+(.*)$", s)
    tail = m.group(1) if m else s

    # strip extension
    tail = re.sub(r"\.(ya?ml)$", "", tail, flags=re.I)

    # strip suffix like __7
    tail = re.sub(r"__\d+$", "", tail)

    return tail.strip()


def match_workflow(candidate_stem: str, workflows: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """
    Prefer match by path stem (.github/workflows/<stem>.yml/.yaml).
    Fallback: workflow name contains stem.
    """
    stem = (candidate_stem or "").lower().strip()
    if not stem:
        return None

    def path_stem(w):
        p = str(w.get("path", ""))
        base = p.split("/")[-1]
        base = re.sub(r"\.(ya?ml)$", "", base, flags=re.I)
        return base.lower()

    exact = [w for w in workflows if path_stem(w) == stem]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        # choose active first
        active = [w for w in exact if str(w.get("state", "")).lower() == "active"]
        return active[0] if active else exact[0]

    # fallback: name contains stem
    contains = [w for w in workflows if stem in str(w.get("name", "")).lower()]
    if len(contains) == 1:
        return contains[0]
    if len(contains) > 1:
        active = [w for w in contains if str(w.get("state", "")).lower() == "active"]
        return active[0] if active else contains[0]

    return None


# -----------------------------
# Main pipeline
# -----------------------------
def run_pipeline(
    ci_csv_path: str = CI_CSV_PATH,
    gmd_csv_path: str = GMD_CSV_PATH,
    out_dir: str = OUTPUT_DIR,
    lookback_days: int = LOOKBACK_DAYS,
):
    logger = setup_logger(out_dir)

    logger.info("Loading CSV inputs...")
    ci_df = pd.read_csv(ci_csv_path)
    gmd_df = pd.read_csv(gmd_csv_path)

    # Basic validation
    req_ci_cols = {"filename", "full_name"}
    req_gmd_cols = {"full_name"}
    if not req_ci_cols.issubset(set(ci_df.columns)):
        raise RuntimeError(f"CI CSV missing columns: {req_ci_cols - set(ci_df.columns)}")
    if not req_gmd_cols.issubset(set(gmd_df.columns)):
        raise RuntimeError(f"GMD CSV missing columns: {req_gmd_cols - set(gmd_df.columns)}")

    # Add candidate workflow stem
    ci_df["candidate_workflow_stem"] = ci_df["filename"].astype(str).map(workflow_stem_from_candidate_filename)

    # Join repo-level gmd info into each workflow candidate
    merged = ci_df.merge(
        gmd_df.drop_duplicates(subset=["full_name"]),
        on="full_name",
        how="left",
        suffixes=("", "_gmd"),
    )

    # Optionally cap repos
    if MAX_REPOS is not None:
        keep_repos = merged["full_name"].drop_duplicates().head(MAX_REPOS).tolist()
        merged = merged[merged["full_name"].isin(keep_repos)].copy()

    tokens = collect_tokens(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens=tokens, logger=logger)

    since_dt = dt.datetime.now(tz=dt.timezone.utc) - dt.timedelta(days=lookback_days)
    since_str = since_dt.date().isoformat()
    logger.info(f"Lookback window: {lookback_days} days (created >= {since_str})")

    # Verify workflows
    verified_rows = []
    workflows_cache: Dict[str, List[Dict[str, Any]]] = {}

    logger.info("Verifying workflows via GitHub API (list workflows per repo + match candidates)...")
    for i, row in merged.iterrows():
        full_name = str(row["full_name"]).strip()
        candidate_file = str(row["filename"])
        stem = str(row["candidate_workflow_stem"])

        if not full_name or "/" not in full_name:
            verified_rows.append({**row.to_dict(), "match_status": "bad_full_name"})
            continue

        if full_name not in workflows_cache:
            try:
                wf_list = gh.get_paginated_items(f"/repos/{full_name}/actions/workflows", "workflows", params={"per_page": PER_PAGE})
            except Exception as e:
                logger.warning(f"[{full_name}] failed to list workflows: {e}")
                workflows_cache[full_name] = []
            else:
                workflows_cache[full_name] = wf_list

        wf_list = workflows_cache[full_name]
        matched = match_workflow(stem, wf_list)

        out = row.to_dict()
        out["candidate_workflow_file"] = candidate_file
        out["candidate_workflow_stem"] = stem

        if matched:
            out["match_status"] = "matched"
            out["workflow_id"] = matched.get("id")
            out["workflow_name"] = matched.get("name")
            out["workflow_path"] = matched.get("path")
            out["workflow_state"] = matched.get("state")
        else:
            out["match_status"] = "unmatched"
            out["workflow_id"] = None
            out["workflow_name"] = None
            out["workflow_path"] = None
            out["workflow_state"] = None

        verified_rows.append(out)

    verified_df = pd.DataFrame(verified_rows)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    verified_path = Path(out_dir) / "verified_workflows.csv"
    verified_df.to_csv(verified_path, index=False, encoding="utf-8")
    logger.info(f"Wrote: {verified_path}")

    matched_df = verified_df[verified_df["match_status"] == "matched"].copy()
    if matched_df.empty:
        logger.warning("No workflows matched. Check candidate stems or repo access permissions.")
        return {
            "verified_df": verified_df,
            "runs_df": pd.DataFrame(),
            "jobs_df": pd.DataFrame(),
            "steps_df": pd.DataFrame(),
            "workflow_metrics_df": pd.DataFrame(),
            "repo_metrics_df": pd.DataFrame(),
        }

    # Fetch runs/jobs/steps
    run_rows = []
    job_rows = []
    step_rows = []

    logger.info("Fetching runs + jobs + steps for matched workflows (this can take a while)...")
    for k, r in matched_df.iterrows():
        repo = str(r["full_name"])
        workflow_id = int(r["workflow_id"])
        workflow_key = f"{repo}::{workflow_id}"

        # list runs
        try:
            runs = gh.get_paginated_items(
                f"/repos/{repo}/actions/workflows/{workflow_id}/runs",
                "workflow_runs",
                params={"created": f">={since_str}", "per_page": PER_PAGE},
                hard_limit=MAX_RUNS_PER_WORKFLOW,
            )
        except Exception as e:
            logger.warning(f"[{workflow_key}] failed to list runs: {e}")
            continue

        for run in runs:
            run_id = run.get("id")
            status = run.get("status")
            conclusion = run.get("conclusion")

            created_at = run.get("created_at")
            updated_at = run.get("updated_at")
            run_started_at = run.get("run_started_at") or run.get("created_at")

            start_dt = parse_iso_z(run_started_at)
            end_dt = parse_iso_z(updated_at)
            dur_s = (end_dt - start_dt).total_seconds() if (start_dt and end_dt) else None

            run_rows.append({
                "repo": repo,
                "workflow_id": workflow_id,
                "workflow_name": r.get("workflow_name"),
                "workflow_path": r.get("workflow_path"),
                "candidate_workflow_file": r.get("candidate_workflow_file"),
                "candidate_workflow_stem": r.get("candidate_workflow_stem"),
                "run_id": run_id,
                "run_number": run.get("run_number"),
                "event": run.get("event"),
                "head_branch": run.get("head_branch"),
                "status": status,
                "conclusion": conclusion,
                "created_at": created_at,
                "run_started_at": run_started_at,
                "updated_at": updated_at,
                "run_duration_s": dur_s,
            })

            # jobs (+ steps)
            try:
                jobs = gh.get_paginated_items(
                    f"/repos/{repo}/actions/runs/{run_id}/jobs",
                    "jobs",
                    params={"per_page": PER_PAGE},
                )
            except Exception as e:
                logger.warning(f"[{repo} run {run_id}] failed to list jobs: {e}")
                continue

            for job in jobs:
                job_id = job.get("id")
                j_start = parse_iso_z(job.get("started_at"))
                j_end = parse_iso_z(job.get("completed_at"))
                j_dur_s = (j_end - j_start).total_seconds() if (j_start and j_end) else None

                job_rows.append({
                    "repo": repo,
                    "workflow_id": workflow_id,
                    "run_id": run_id,
                    "job_id": job_id,
                    "job_name": job.get("name"),
                    "job_status": job.get("status"),
                    "job_conclusion": job.get("conclusion"),
                    "job_started_at": job.get("started_at"),
                    "job_completed_at": job.get("completed_at"),
                    "job_duration_s": j_dur_s,
                    "runner_name": job.get("runner_name"),
                    "runner_group_name": job.get("runner_group_name"),
                    "labels": ";".join(job.get("labels") or []),
                })

                steps = job.get("steps") or []
                for step in steps:
                    s_start = parse_iso_z(step.get("started_at"))
                    s_end = parse_iso_z(step.get("completed_at"))
                    s_dur_s = (s_end - s_start).total_seconds() if (s_start and s_end) else None

                    step_rows.append({
                        "repo": repo,
                        "workflow_id": workflow_id,
                        "run_id": run_id,
                        "job_id": job_id,
                        "step_number": step.get("number"),
                        "step_name": step.get("name"),
                        "step_status": step.get("status"),
                        "step_conclusion": step.get("conclusion"),
                        "step_started_at": step.get("started_at"),
                        "step_completed_at": step.get("completed_at"),
                        "step_duration_s": s_dur_s,
                    })

    runs_df = pd.DataFrame(run_rows)
    jobs_df = pd.DataFrame(job_rows)
    steps_df = pd.DataFrame(step_rows)

    # Write raw extracts
    runs_path = Path(out_dir) / "runs_60d.csv"
    jobs_path = Path(out_dir) / "jobs_60d.csv"
    steps_path = Path(out_dir) / "steps_60d.csv"
    runs_df.to_csv(runs_path, index=False, encoding="utf-8")
    jobs_df.to_csv(jobs_path, index=False, encoding="utf-8")
    steps_df.to_csv(steps_path, index=False, encoding="utf-8")
    logger.info(f"Wrote: {runs_path}")
    logger.info(f"Wrote: {jobs_path}")
    logger.info(f"Wrote: {steps_path}")

    # -----------------------------
    # Metrics
    # -----------------------------
    logger.info("Computing metrics (workflow-level and repo-level)...")

    def pct(x: float) -> float:
        return round(100.0 * x, 3)

    # Only completed runs for duration stats
    runs_done = runs_df.copy()
    runs_done["run_duration_s"] = runs_done["run_duration_s"].map(safe_float)
    runs_done = runs_done[runs_done["status"].astype(str).str.lower() == "completed"].copy()

    # Workflow key: repo + workflow_id (since a repo can have multiple workflows)
    group_cols = ["repo", "workflow_id", "workflow_name", "workflow_path", "candidate_workflow_file", "candidate_workflow_stem"]

    def quantile_s(series, q):
        series = pd.to_numeric(series, errors="coerce").dropna()
        if series.empty:
            return None
        return float(series.quantile(q))

    wf_rows = []
    for key, g in runs_done.groupby(group_cols, dropna=False):
        repo, wf_id, wf_name, wf_path, cand_file, cand_stem = key
        total = len(g)
        success = int((g["conclusion"].astype(str).str.lower() == "success").sum())
        fail = int((g["conclusion"].astype(str).str.lower() == "failure").sum())
        cancelled = int((g["conclusion"].astype(str).str.lower() == "cancelled").sum())

        dur_p50 = quantile_s(g["run_duration_s"], 0.50)
        dur_p90 = quantile_s(g["run_duration_s"], 0.90)
        dur_mean = float(pd.to_numeric(g["run_duration_s"], errors="coerce").dropna().mean()) if g["run_duration_s"].notna().any() else None

        wf_rows.append({
            "repo": repo,
            "workflow_id": wf_id,
            "workflow_name": wf_name,
            "workflow_path": wf_path,
            "candidate_workflow_file": cand_file,
            "candidate_workflow_stem": cand_stem,
            "runs_completed": total,
            "runs_success": success,
            "runs_failure": fail,
            "runs_cancelled": cancelled,
            "success_rate_pct": pct(success / total) if total else None,
            "run_duration_p50_min": to_minutes(dur_p50),
            "run_duration_p90_min": to_minutes(dur_p90),
            "run_duration_mean_min": to_minutes(dur_mean),
        })

    workflow_metrics_df = pd.DataFrame(wf_rows)

    # Repo-level metrics (across all matched workflows)
    repo_rows = []
    for repo, g in runs_done.groupby(["repo"], dropna=False):
        total = len(g)
        success = int((g["conclusion"].astype(str).str.lower() == "success").sum())
        dur_p50 = quantile_s(g["run_duration_s"], 0.50)
        dur_p90 = quantile_s(g["run_duration_s"], 0.90)
        repo_rows.append({
            "repo": repo,
            "runs_completed": total,
            "runs_success": success,
            "success_rate_pct": pct(success / total) if total else None,
            "run_duration_p50_min": to_minutes(dur_p50),
            "run_duration_p90_min": to_minutes(dur_p90),
        })

    repo_metrics_df = pd.DataFrame(repo_rows)

    # Enrich metrics with signals (join using repo + candidate_workflow_file for workflow-level)
    # This keeps your CI/GMD signal columns attached to metrics.
    # (If there are duplicates, we keep the first occurrence.)
    enrich_cols = [c for c in verified_df.columns if c not in ("workflow_id", "workflow_name", "workflow_path", "workflow_state")]
    enrich_unique = verified_df[enrich_cols].drop_duplicates(subset=["full_name", "candidate_workflow_file"], keep="first").copy()
    enrich_unique = enrich_unique.rename(columns={"full_name": "repo"})

    workflow_metrics_df = workflow_metrics_df.merge(
        enrich_unique,
        on=["repo", "candidate_workflow_file"],
        how="left",
    )

    # Write metrics
    wf_metrics_path = Path(out_dir) / "workflow_metrics_60d.csv"
    repo_metrics_path = Path(out_dir) / "repo_metrics_60d.csv"
    workflow_metrics_df.to_csv(wf_metrics_path, index=False, encoding="utf-8")
    repo_metrics_df.to_csv(repo_metrics_path, index=False, encoding="utf-8")
    logger.info(f"Wrote: {wf_metrics_path}")
    logger.info(f"Wrote: {repo_metrics_path}")

    logger.info("DONE.")
    logger.info(f"Matched workflows: {len(matched_df)} / {len(verified_df)} candidates")
    logger.info(f"Runs fetched (raw): {len(runs_df)} | Jobs: {len(jobs_df)} | Steps: {len(steps_df)}")

    return {
        "verified_df": verified_df,
        "runs_df": runs_df,
        "jobs_df": jobs_df,
        "steps_df": steps_df,
        "workflow_metrics_df": workflow_metrics_df,
        "repo_metrics_df": repo_metrics_df,
    }


# -----------------------------
# Run now (Jupyter-friendly)
# -----------------------------
results = run_pipeline(
    ci_csv_path=CI_CSV_PATH,
    gmd_csv_path=GMD_CSV_PATH,
    out_dir=OUTPUT_DIR,
    lookback_days=LOOKBACK_DAYS,
)

# Quick peek
display(results["workflow_metrics_df"].head(10))
display(results["repo_metrics_df"].head(10))

